# Data Loading

We load a mixed knowledge base and a separate evaluation set.

**Knowledge Base** (retrieved from during RAG):
- MedQuAD — consumer Q&A from NIH sources
- MedlinePlus — official NLM health topics
- PubMedQA artificial — research abstracts (context field)
- MedRedQA — consumer questions with physician answers

**Evaluation Set** (only used to score the system):
- PubMedQA labeled — 1,000 expert-annotated questions
- HealthSearchQA — 4,436 real consumer questions

Output: raw CSVs in `data/raw/`.

In [1]:
import os
import io
import re
import ast
import zipfile
import requests
import pandas as pd
from pathlib import Path
from xml.etree import ElementTree as ET
from datasets import load_dataset

In [2]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
REPORTS_DIR = ROOT / "reports"
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. MedQuAD

Consumer medical Q&A originally from 12 NIH sites. The Kaggle mirror already filters out rows with empty answers.

In [3]:
ds = load_dataset("keivalya/MedQuad-MedicalQnADataset")
df_medquad = ds[list(ds.keys())[0]].to_pandas()

print("rows:", len(df_medquad))
print("cols:", df_medquad.columns.tolist())

rows: 16407
cols: ['qtype', 'Question', 'Answer']


In [4]:
pd.set_option("display.max_colwidth", 120)
df_medquad.head(2)

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?,"LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected roden..."
1,symptoms,What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?,"LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symp..."


## 2. MedlinePlus (NLM XML)

NLM publishes health topics as a daily XML file. We scrape the page to find the current filename, then download and parse it.

In [5]:
html = requests.get("https://medlineplus.gov/xml.html", timeout=30).text

zip_url = re.findall(
    r"https://medlineplus\.gov/xml/mplus_topics_compressed_\d{4}-\d{2}-\d{2}\.zip",
    html,
)[0]
print(zip_url)

https://medlineplus.gov/xml/mplus_topics_compressed_2026-10-03.zip


In [6]:
r = requests.get(zip_url, timeout=120)
z = zipfile.ZipFile(io.BytesIO(r.content))
xml_name = [n for n in z.namelist() if n.endswith(".xml")][0]
xml_bytes = z.read(xml_name)

root = ET.fromstring(xml_bytes)

rows = []
for topic in root.findall("health-topic"):
    if topic.get("language") != "English":
        continue
    summary_el = topic.find("full-summary")
    summary = "".join(summary_el.itertext()).strip() if summary_el is not None else ""
    rows.append({
        "title": topic.get("title", ""),
        "url": topic.get("url", ""),
        "summary": summary,
    })

df_medline = pd.DataFrame(rows)
print("rows:", len(df_medline))
df_medline.head(2)

rows: 1017


,title,url,summary
0,A1C,https://medlineplus.gov/a1c.html,"<p>A1C is a blood test for <a href=""https://medlineplus.gov/diabetestype2.html"">type 2 diabetes</a> and <a href=""htt..."
1,Abdominal Pain,https://medlineplus.gov/abdominalpain.html,"<p>Your abdomen extends from below your chest to your groin. Some people call it the stomach, but your abdomen conta..."


## 3. PubMedQA artificial (KB)

211K records. The `context` field holds the PubMed abstract — that's what we keep for retrieval. The question/answer pairs are Boolean (yes/no/maybe) and are not useful on their own.

In [7]:
ds = load_dataset("qiaojin/PubMedQA", "pqa_artificial")
df_pubmed_kb = ds["train"].to_pandas()

print("rows:", len(df_pubmed_kb))
print("cols:", df_pubmed_kb.columns.tolist())

rows: 211269
cols: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


In [8]:
# context is a dict with a 'contexts' list. Join into a single string.
def parse_context(x):
    if isinstance(x, dict):
        return " ".join(x.get("contexts", []))
    try:
        d = ast.literal_eval(x)
        if isinstance(d, dict):
            return " ".join(d.get("contexts", []))
    except (ValueError, SyntaxError):
        pass
    return str(x)

df_pubmed_kb["context"] = df_pubmed_kb["context"].apply(parse_context)
print("sample:", df_pubmed_kb["context"].iloc[0][:200])

sample: Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated 


## 4. MedRedQA (KB)

Consumer questions from `/r/askdocs`. We keep only answers written by verified physicians.

In [9]:
medred = load_dataset("bagga005/medredqa")

parts = [medred[s].to_pandas() for s in medred.keys()]
df_medred = pd.concat(parts, ignore_index=True)

print("raw rows:", len(df_medred))
print("cols:", df_medred.columns.tolist())

raw rows: 50991
cols: ['id', 'title', 'body', 'response', 'response_score', 'occupation', 'pmcids']


In [10]:
# Keep answers written by physicians only
physician_keywords = ["MD", "Physician", "Doctor", "DO"]
mask = df_medred["occupation"].astype(str).str.contains(
    "|".join(physician_keywords), case=False, na=False
)

df_medred = df_medred[mask].reset_index(drop=True)
print("after filter:", len(df_medred))

after filter: 40602


In [11]:
pd.set_option("display.max_colwidth", 120)
df_medred[["title", "body", "response", "occupation"]].head(3)

,title,body,response,occupation
0,I need some opinions on this please!,"Hello i am a male 20 years old, 175 cm height and 48 kilograms, I quit smoking a year ago no other medical issues ex...",Sounds like IBS. Prepare yourself for that colonoscopy to be normal.,MD - Peds GI | Top Contributor
1,First experience with Contact dermatitis,"Hello \n\n33 year old Male , Hispanic. \n\nI encountered my first experience with Contact dermatitis from a reaction...",Whenever the skin is inflamed it scales during or afterwards.,Physician - Dermatologist | Top Contributor
2,Continuing Amoxicillin w/ Augmentin,"I am closing in on the last few days of a 7-day, 825mg/2x/day regimen of Amoxicillin and think I might need a few mo...",Why are you on antibiotics?\nSome conditions may not need antibiotics and fewer conditions require extending antibio...,Physician


## 5. PubMedQA labeled (Eval)

1,000 expert-annotated questions with gold yes/no/maybe labels. Used as a research-style benchmark, not as KB.

In [12]:
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
df_pubmed_eval = ds["train"].to_pandas()

print("rows:", len(df_pubmed_eval))
print("cols:", df_pubmed_eval.columns.tolist())

rows: 1000
cols: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


## 6. HealthSearchQA (Eval)

Consumer questions from Google search autocomplete. No answers — used to measure coverage.

In [13]:
ds = load_dataset("aisc-team-d1/healthsearchqa")
df_health = ds[list(ds.keys())[0]].to_pandas()

print("rows:", len(df_health))
print("cols:", df_health.columns.tolist())

rows: 4436
cols: ['id', 'question']


In [14]:
df_health.head(5)

,id,question
0,1.0,Are benign brain tumors serious?
1,2.0,Are boils and carbuncles curable?
2,3.0,Are bone cysts serious?
3,4.0,Are cold sores a herpes virus?
4,5.0,Are dental abscesses serious?


## 7. Save raw copies

In [15]:
files = {
    # KB
    "medquad_raw.csv": df_medquad,
    "medlineplus_raw.csv": df_medline,
    "pubmedqa_kb_raw.csv": df_pubmed_kb,
    "medredqa_raw.csv": df_medred,
    # Eval
    "pubmedqa_eval_raw.csv": df_pubmed_eval,
    "healthsearchqa_raw.csv": df_health,
}

for name, frame in files.items():
    path = RAW_DIR / name
    frame.to_csv(path, index=False)
    print(f"{name}: {len(frame):,} rows, {os.path.getsize(path) / 1024**2:.1f} MB")

medquad_raw.csv: 16,407 rows, 21.4 MB
medlineplus_raw.csv: 1,017 rows, 2.8 MB
pubmedqa_kb_raw.csv: 211,269 rows, 356.2 MB
medredqa_raw.csv: 40,602 rows, 54.8 MB
pubmedqa_eval_raw.csv: 1,000 rows, 2.2 MB
healthsearchqa_raw.csv: 4,436 rows, 0.1 MB


## 8. Report

In [16]:
kb_total = len(df_medquad) + len(df_medline) + len(df_pubmed_kb) + len(df_medred)
eval_total = len(df_pubmed_eval) + len(df_health)

report = f"""# Data Sources

## Knowledge Base ({kb_total:,} records total)

| Source | Rows | Type |
|--------|------|------|
| MedQuAD | {len(df_medquad):,} | Consumer Q&A |
| MedlinePlus | {len(df_medline):,} | Official health topics |
| PubMedQA (context only) | {len(df_pubmed_kb):,} | Research abstracts |
| MedRedQA (physician answers) | {len(df_medred):,} | Consumer Q&A |

## Evaluation ({eval_total:,} questions total)

| Source | Rows | Type |
|--------|------|------|
| PubMedQA labeled | {len(df_pubmed_eval):,} | Research benchmark |
| HealthSearchQA | {len(df_health):,} | Consumer questions |
"""

(REPORTS_DIR / "data_sources.md").write_text(report)
print(report)

# Data Sources

## Knowledge Base (269,295 records total)

| Source | Rows | Type |
|--------|------|------|
| MedQuAD | 16,407 | Consumer Q&A |
| MedlinePlus | 1,017 | Official health topics |
| PubMedQA (context only) | 211,269 | Research abstracts |
| MedRedQA (physician answers) | 40,602 | Consumer Q&A |

## Evaluation (5,436 questions total)

| Source | Rows | Type |
|--------|------|------|
| PubMedQA labeled | 1,000 | Research benchmark |
| HealthSearchQA | 4,436 | Consumer questions |



Next: `02_preprocessing.ipynb`